<a href="https://colab.research.google.com/github/trinhtattran/RAGassistant/blob/main/PD_RAG_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_DIR = "/content/drive/MyDrive/RAG/data"

In [ ]:
!pip -q install langchain langchain-community sentence-transformers faiss-cpu pypdf

In [ ]:
!pip install -q pypdf==3.17.4

In [ ]:
!pip install -q pdfplumber


In [ ]:
import langchain
import sentence_transformers
import faiss
import pypdf

print("All imports OK")


In [ ]:
import os, re
from langchain_community.document_loaders import PDFPlumberLoader #more tables on my docs now

DATA_DIR = "/content/drive/MyDrive/RAG/data"

def load_pdfs(folder_path):
    docs = []
    for fn in os.listdir(folder_path):
        if fn.lower().endswith(".pdf"):
            loader = PDFPlumberLoader(os.path.join(folder_path, fn))
            loaded = loader.load()
            for d in loaded:
                d.metadata["source"] = fn
            docs.extend(loaded)
    return docs

def basic_clean(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

raw_docs = load_pdfs(DATA_DIR)

for d in raw_docs:
    d.page_content = basic_clean(d.page_content)

print("Loaded pages:", len(raw_docs))
print("Example source:", raw_docs[0].metadata)
print("Example text:", raw_docs[0].page_content[:400])


In [ ]:
print("Loaded pages:", len(raw_docs))
print("Sample text length:", len(raw_docs[0].page_content))
print(raw_docs[0].page_content[:500])
#testing to see if docs are read

In [ ]:
from collections import defaultdict
import numpy as np

# raw_docs is a list of LangChain Documents (one per page)
lengths = [len(d.page_content.strip()) for d in raw_docs]

print("Total pages:", len(raw_docs))
print("Pages with 0 chars:", sum(l == 0 for l in lengths))
print("Pages with <50 chars:", sum(l < 50 for l in lengths))
print("Median chars/page:", int(np.median(lengths)))
print("10th percentile chars/page:", int(np.percentile(lengths, 10)))

# Group by PDF filename
by_pdf = defaultdict(list)
for d in raw_docs:
    by_pdf[d.metadata.get("source","UNKNOWN")].append(len(d.page_content.strip()))

print("\nWorst PDFs by median extracted chars/page:")
stats = []
for pdf, lens in by_pdf.items():
    stats.append((pdf, int(np.median(lens)), sum(l == 0 for l in lens), len(lens)))
stats.sort(key=lambda x: x[1])  # sort by median text length

for pdf, med, zeros, total in stats[:10]:
    print(f"{pdf:45}  median={med:4d}  zero_pages={zeros:3d}/{total}")


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("One chunk metadata:", chunks[0].metadata)
print("One chunk text:", chunks[0].page_content[:300]) #need to print out to see


In [ ]:
import random

def print_random_chunks(chunks, n=5, max_chars=900):
    for idx in random.sample(range(len(chunks)), n):
        c = chunks[idx]
        print("\n" + "="*90)
        print(f"CHUNK #{idx}")
        print("SOURCE:", c.metadata.get("source"), "| PAGE:", c.metadata.get("page"))
        print(c.page_content[:max_chars])

print_random_chunks(chunks, n=5)
#this size works!

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# shared cohort model (recommended in spec)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# build the vector store (embeds all chunks once)
db = FAISS.from_documents(chunks, embeddings)

print("FAISS index built.")
print("Total chunks indexed:", len(chunks))


In [ ]:
def retrieve_top_k(query: str, k: int = 6):
    hits = db.similarity_search(query, k=k)  # returns LangChain Documents
    # each hit has .page_content and .metadata (source/page)
    return hits

def show_hits(hits, max_chars=450):
    for i, h in enumerate(hits, 1):
        print("\n" + "-"*90)
        print(f"HIT {i} | source={h.metadata.get('source')} | page={h.metadata.get('page')}")
        print(h.page_content[:max_chars])


In [ ]:
q = "What are red flags for atypical parkinsonism?"
hits = retrieve_top_k(q, k=6)
show_hits(hits)


In [ ]:
FAISS_DIR = "/content/drive/MyDrive/RAG/faiss_index"
db.save_local(FAISS_DIR)
print("Saved FAISS index to:", FAISS_DIR)

# Later, reload like this:
# db = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)


In [ ]:
test_questions = [
  "What are red flags that suggest atypical parkinsonism rather than idiopathic Parkinson's disease?",
  "How does essential tremor differ from Parkinson's disease tremor clinically?",
  "What clinical features distinguish progressive supranuclear palsy (PSP) from Parkinson's disease?",
  "What clinical features distinguish multiple system atrophy (MSA) from Parkinson's disease?",
  "What is drug-induced parkinsonism and how does it present compared to Parkinson's disease?",
  "What are common causes of secondary parkinsonism?",
  "What does bradykinesia mean clinically and how is it assessed?",
  "What does rigidity mean clinically and how is it assessed?",
  "What does the MDS-UPDRS Part III measure?",
  "What is Hoehn and Yahr staging used for?",
  "What non-motor symptoms are commonly associated with Parkinson's disease?",
  "When is dopamine transporter imaging (DaTscan) considered in evaluation?",
  "What are limitations of DaTscan/dopamine transporter imaging?",
  "What features suggest corticobasal syndrome rather than Parkinson's disease?",
  "What features suggest vascular parkinsonism rather than Parkinson's disease?",
  "What is the typical progression pattern of idiopathic Parkinson's disease?",
  "What gait abnormalities are described in Parkinson's disease?",
  "What does postural instability imply in parkinsonism evaluation?",
  "What early autonomic symptoms may suggest atypical parkinsonism?",
  "What does poor response to levodopa suggest about the diagnosis?"
]
print("Total questions:", len(test_questions))


In [ ]:
for q in test_questions[:10]:
    print("\n" + "="*100)
    print("Q:", q)
    hits = retrieve_top_k(q, k=6)
    # FAISS returns Documents
    for i, h in enumerate(hits[:3], 1):
        print(f"TOP {i}: {h.metadata.get('source')} p{h.metadata.get('page')}")
        print(h.page_content[:220].replace("\n"," "), "...")
